# <p style="background-color:red;color:white;font-family:cursive ;font-size:110%;text-align:center;border-radius: 15px 50px;">1. Introduction</p>

In this notebook, we embark on a comprehensive journey to enhance the predictive power of our model through advanced feature engineering and ensemble learning. The ensemble comprises well-known gradient boosting algorithms such as CATBoost, XGBoost, and LGBM, each contributing its unique strengths to the overall predictive capability. 

### Feature Engineering
* We begin by exploring and applying the best feature engineering practices found in public notebooks for this competition, the aim is to extract maximum information from our dataset. 
* Please find the credit and references for each notebook that inspired the feature used below.

### Pipelines
* We will then begin to build and assemble the pipeline of data-processing & engineered features.

### Ensemble Learning
* Subsequently, we delve into the world of ensemble learning, combining the strengths of different models with varying weights. 
* The evaluation is centered around the ROC_AUC score, and we visually dissect the model's performance using Confusion Matrix insights. 

### Tuning using Optuna
* To further elevate our model's performance, we employ Optuna for hyperparameter optimization, focusing not only on weight adjustments but also fine-tuning various parameters for each individual model. 
* The goal is to achieve an optimized and robust predictive model ready to tackle the challenges of our dataset.

# <p style="background-color:red ;color:white;font-family:cursive ;font-size:110%;text-align:center;border-radius: 15px 50px;">2. Import + Load Data</p>

In [ ]:
#Basic libraries for EDA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#sklearn library
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler,RobustScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import FeatureUnion
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold, KFold
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import  confusion_matrix
from sklearn.metrics import precision_score, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import  cross_val_score #Hyperparameter Tuning


#xgboost library
import xgboost as xgb
from xgboost import plot_importance
from xgboost import XGBClassifier

#lgb library
import lightgbm as lgb
from lightgbm import LGBMClassifier

#catboost library
from catboost import CatBoostClassifier

import optuna


import warnings
# Set global warning filter
warnings.filterwarnings("ignore")
# Suppress LightGBM warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*num_leaves.*")
warnings.filterwarnings("ignore", category=UserWarning, message=".*No further splits with positive gain.*")

Following suggestion from @Hriday, to consider to utilise original dataset to expand training dataset.

In [ ]:
sample = pd.read_csv('/kaggle/input/playground-series-s4e1/sample_submission.csv')
train = pd.read_csv('/kaggle/input/playground-series-s4e1/train.csv')
original=pd.read_csv('/kaggle/input/bank-customer-churn-prediction/Churn_Modelling.csv')
test = pd.read_csv('/kaggle/input/playground-series-s4e1/test.csv')

train.drop(columns=["id"],inplace=True)
test.drop(columns=["id"],inplace=True)
original.drop(columns=['RowNumber'],inplace=True)

train_copy=train.copy()
test_copy=test.copy()
original_copy=original.copy()

original["original"]=1
train["original"]=0
test["original"]=0

train=pd.concat([train,original.dropna()],axis=0)
train.reset_index(inplace=True,drop=True)

trainX = train.drop(['Exited'], axis=1)
trainy = train['Exited']

# <p style="background-color:red ;color:white;font-family:cursive ;font-size:110%;text-align:center;border-radius: 15px 50px;">3. Feature Engineering</p>


## Age Binning

In [ ]:
class AgeBinning(BaseEstimator, TransformerMixin):
    def __init__(self, n_bins):
        self.n_bins = n_bins
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        transformed = pd.qcut(X['Age'], self.n_bins, labels=False)
        transformed[transformed.isna()] = 0
        transformed_series = pd.Series(transformed, 
                                       name=f'QCut{self.n_bins}_Age',
                                       index=X.index)
        X_copy = X.copy()
        return pd.concat([X_copy, transformed_series], axis=1)
        
AgeBinning(5).fit_transform(train)

## Credit Score Binning

In [ ]:
class CreditScoreBinning(BaseEstimator, TransformerMixin):
    def __init__(self, n_bins):
        self.n_bins = n_bins
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        transformed = pd.qcut(X['CreditScore'], self.n_bins, labels=False)
        transformed[transformed.isna()] = 0
        transformed_series = pd.Series(transformed, 
                                       name=f'QCut{self.n_bins}_CreditScore',
                                       index=X.index)
        X_copy = X.copy()
        return pd.concat([X_copy, transformed_series], axis=1)
        
CreditScoreBinning(5).fit_transform(train)

## Estimated Salary Binning

In [ ]:
class SalaryBinning(BaseEstimator, TransformerMixin):
    def __init__(self, n_bins):
        self.n_bins = n_bins
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        transformed = pd.qcut(X['EstimatedSalary'], self.n_bins, labels=False)
        transformed[transformed.isna()] = 0
        transformed_series = pd.Series(transformed, 
                                       name=f'QCut{self.n_bins}_Est_Salary',
                                       index=X.index)
        X_copy = X.copy()
        return pd.concat([X_copy, transformed_series], axis=1)
        
SalaryBinning(10).fit_transform(train)

## Balance to Salary Ratio
referenced from: https://www.kaggle.com/code/ashishkumarak/playground-s4e1-bank-churn-prediction-eda#%F0%9F%92%BD-Importing-the-data

In [ ]:
class BalanceSalaryRatioTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        X_copy['Balance_Salary_Ratio'] = X_copy['Balance'] / X_copy['EstimatedSalary']
        
        return X_copy

BalanceSalaryRatioTransformer().fit_transform(train)        

## Geography and Gender interaction
referenced from: https://www.kaggle.com/code/ashishkumarak/playground-s4e1-bank-churn-prediction-eda#%F0%9F%92%BD-Importing-the-data

In [ ]:
class GeoGenderTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        X_copy['Geo_Gender'] = X_copy['Geography'] + '_' + X_copy['Gender']
        return X_copy
    
GeoGenderTransformer().fit_transform(train)   

## Total Prouducts Used
referenced from: https://www.kaggle.com/code/ashishkumarak/playground-s4e1-bank-churn-prediction-eda#%F0%9F%92%BD-Importing-the-data

In [ ]:
class TotalProductsTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        X_copy['Total_Products_Used'] = X_copy['NumOfProducts'] + X_copy['HasCrCard']
        return X_copy

TotalProductsTransformer().fit_transform(train)   

## Gender and Total Product interaction
referenced from: https://www.kaggle.com/code/ashishkumarak/playground-s4e1-bank-churn-prediction-eda#%F0%9F%92%BD-Importing-the-data

In [ ]:
class TpGenderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, total_products_field='Total_Products_Used', gender_field='Gender'):
        self.total_products_field = total_products_field
        self.gender_field = gender_field
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        X_copy['Tp_Gender'] = X_copy[self.total_products_field].astype('str') + X_copy[self.gender_field]
        return X_copy

train_tp = TotalProductsTransformer().fit_transform(train)   
TpGenderTransformer().fit_transform(train_tp)    

## TFIDF-PCA (Text Transformation)
referenced from: https://www.kaggle.com/code/arunklenin/ps4e1-advanced-feature-engineering-ensemble#TFIDF-PCA-(Text-Transformation)

In [ ]:
class TFIDFPCATransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column, max_features, n_components):
        self.column = column
        self.max_features = max_features
        self.n_components = n_components
        self.vectorizer = TfidfVectorizer(max_features=max_features)
        self.svd = TruncatedSVD(n_components=n_components)

    def fit(self, X, y=None):
        vectors = self.vectorizer.fit_transform(X[self.column])
        self.svd.fit(vectors)
        return self

    def transform(self, X):
        vectors = self.vectorizer.transform(X[self.column])
        svd_result = self.svd.transform(vectors)

        tfidf_df = pd.DataFrame(svd_result, columns=[f"{self.column}_tfidf_{i}" for i in range(self.n_components)])
        X = pd.concat([X, tfidf_df], axis="columns")
        return X
    
TFIDFPCATransformer(column="Surname", max_features=1000, n_components=5).fit_transform(train) 

## k-Means Clusterer
with reference: https://www.kaggle.com/code/arunklenin/ps4e1-advanced-feature-engineering-ensemble#4.3-Numerical-Clustering

In [ ]:
class KMeansClusterer(BaseEstimator, TransformerMixin):
    def __init__(self, features, n_clusters=20, random_state=0, n_components=None):
        self.features = features
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.n_components = n_components
        self.kmeans = KMeans(n_clusters=n_clusters, n_init=50, random_state=random_state)
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components)
    
    def fit(self, X, y=None):
        X_scaled = self.scaler.fit_transform(X.loc[:, self.features])
        if self.n_components is not None:
            X_scaled = self.pca.fit_transform(X_scaled)
        self.kmeans.fit(X_scaled)
        return self
    
    def transform(self, X):
        X_scaled = self.scaler.transform(X.loc[:, self.features])
        
        # Check for NaN values and replace them with zeros or appropriate values
        if np.isnan(X_scaled).any():
            X_scaled = np.nan_to_num(X_scaled)
        
        if self.n_components is not None:
            X_scaled = self.pca.transform(X_scaled)
        
        X_new = pd.DataFrame()
        X_new["Cluster"] = self.kmeans.predict(X_scaled)
        
        X_copy = X.copy()
        # Convert the "Cluster" column to dense format
        X_new["Cluster"] = X_new["Cluster"].values
        return pd.concat([X_copy.reset_index(drop=True), X_new.reset_index(drop=True)], axis=1)


In [ ]:
#With PCA (specify the number of components, e.g., 3)
clusterer_with_pca = KMeansClusterer(features=["CustomerId","EstimatedSalary","Balance"], n_clusters=10, random_state=123, n_components=3)

# Fit and transform your data
clusterer_with_pca.fit_transform(train)

In [ ]:
#Visualising the clusters
from mpl_toolkits.mplot3d import Axes3D

# Assuming X_train is your training data
X_train_selected = train[["CustomerId","EstimatedSalary","Balance"]]

# Fit and transform with PCA and KMeansClusterer
clusterer_with_pca.fit(X_train_selected)
X_transformed = clusterer_with_pca.transform(X_train_selected)

# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot of data points colored by cluster
scatter = ax.scatter(X_transformed["CustomerId"], X_transformed["EstimatedSalary"], X_transformed["Balance"], c=X_transformed["Cluster"], cmap='viridis', s=50)

# Add labels and title
ax.set_xlabel("CustomerId")
ax.set_ylabel("EstimatedSalary")
ax.set_zlabel('Balance')
ax.set_title('3D Scatter Plot of Clusters')

# Add a colorbar
cbar = plt.colorbar(scatter)
cbar.set_label('Cluster')

plt.show()

## Zero Balance Indicator

In [ ]:
class BalanceTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Create a binary column indicating if the bank account is zero
        X['Balance_is_zero'] = (X['Balance'] == 0).astype(int)
        
               
        Balance_is_zero_series = pd.Series(X['Balance_is_zero'], 
                                           name='Balance_is_zero',
                                           index=X.index)
        return X
            
BalanceTransformer().fit_transform(train)

# <p style="background-color:red ;color:white;font-family:cursive ;font-size:110%;text-align:center;border-radius: 15px 50px;">4. Assembling the Pipeline</p>

## Column Transformer

In [ ]:
#applies transformers to different columns.
multicolumn_prep = ColumnTransformer([ ('encode', 
                                       OneHotEncoder(handle_unknown='ignore'), 
                                       ['Gender', 'Geography','NumOfProducts','HasCrCard','IsActiveMember','Geo_Gender','Tp_Gender']),
                                     ],
                                     remainder='passthrough')
multicolumn_prep

## Drop Columns

In [ ]:
class DropColumn(BaseEstimator, TransformerMixin):
    def __init__(self, cols=[]):
        self.cols = cols
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        
        return X.drop(self.cols, axis=1)

In [ ]:
named_preprocessing_pipeline = Pipeline([
    ('kmeans', KMeansClusterer(features=["CustomerId","EstimatedSalary","Balance"], n_clusters=20, random_state=123, n_components=3)),
    ('surname_tfid', TFIDFPCATransformer(column="Surname", max_features=1000, n_components=5)),
    ('age_binning', AgeBinning(5)),
    ('salary_binning', SalaryBinning(5)),
    ('CS_binning', CreditScoreBinning(5)),    
    ('zero_balance', BalanceTransformer()),
    ('balance_salary_ratio', BalanceSalaryRatioTransformer()),
    ('geo_gender', GeoGenderTransformer()),
    ('total_products', TotalProductsTransformer()),
    ('tp_gender', TpGenderTransformer()),
    ('drop', DropColumn(cols=['CustomerId','Surname'])),
    ('prep',ColumnTransformer([ ('encode', 
                                 OneHotEncoder(handle_unknown='ignore',sparse_output=False), 
                                 ['Gender', 'Geography','NumOfProducts','HasCrCard','IsActiveMember','Geo_Gender','Tp_Gender']),
                              ],
                              remainder='passthrough').set_output(transform='pandas')),
])

named_preprocessing_pipeline                               

In [ ]:
#checking output on train df
df_train = named_preprocessing_pipeline.fit_transform(train.drop(['Exited'], axis=1))
df_train.info()

In [ ]:
#checking output on test df
df_test = pd.DataFrame(named_preprocessing_pipeline.transform(test))
df_test.info()

In [ ]:
# Using Standard Scaler or Robust Scaler to scale numeric variables

class StandardScalerNamed(StandardScaler, TransformerMixin):
    def get_feature_names_out(self, X, y=None):
        return X.columns.tolist()

    def transform(self, X, y=None):
        transformed = super().transform(X, y)
        return pd.DataFrame(transformed, columns=X.columns)
    
    
class RobustScalerNamed(RobustScaler, TransformerMixin):
    def get_feature_names_out(self, X, y=None):
        return X.columns.tolist()

    def transform(self, X, y=None):
        transformed = super().transform(X, y)
        return pd.DataFrame(transformed, columns=X.columns)

In [ ]:
modelling_pipeline = Pipeline(named_preprocessing_pipeline.steps + [('scale',RobustScaler().set_output(transform='pandas')),])
modelling_pipeline

In [ ]:
# modelling_pipeline.fit_transform(train.drop(['Exited'], axis=1))

# <p style="background-color:red ;color:white;font-family:cursive ;font-size:110%;text-align:center;border-radius: 15px 50px;">5. Setting up the Models</p>

# XGBoost Classifier

In [ ]:
X = train.drop(['Exited'], axis=1) 
y = train['Exited']

In [ ]:
# #XGBoost parameters
# xgb_params_1 = {'max_depth': 5,
#  'min_child_weight': 2, 
#  'learning_rate': 0.07353564842520434,
#  'n_estimators': 463, 
#  'subsample': 0.8131149969184862,
#  'colsample_bytree': 0.6598001508811656,
#  'random_state': 42}


#XGBoost best parameters 
xgb_params_optuna = {'max_depth': 5,
                     'min_child_weight': 2, 
                     'learning_rate': 0.07353564842520434,
                     'n_estimators': 463, 
                     'subsample': 0.8131149969184862,
                     'colsample_bytree': 0.6598001508811656,
                     'random_state': 42}

# XGBoost model
xgb_model = XGBClassifier(**xgb_params_optuna)

xgb_pipeline = make_pipeline(modelling_pipeline, xgb_model)
xgb_pipeline

In [ ]:
# # number of folds
# n_splits = 10

# #  StratifiedKFold
# stratkf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# #  cross-validation results
# cv_results = []

# # stratified k-fold cross-validation
# for fold, (train_idx, val_idx) in enumerate(stratkf.split(X, y)):
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y[train_idx], y[val_idx]

#     xgb_pipeline.fit(X_train, y_train )

#     # predictions on the validation set
#     y_val_pred_prob = xgb_pipeline.predict(X_val)
#     y_pred = xgb_pipeline.predict(X_val)
        
#     f1=  f1_score(y_val, y_pred, average='weighted')

#     # Evaluating the model
    
#     roc_auc = roc_auc_score(y_val, y_val_pred_prob)
#     print(f'Fold {fold + 1}, AUC Score on Validation Set: {roc_auc}')
#     print(f'Fold {fold + 1}, F1 Score on Validation Set: {f1}')
#     print('-'*70)

#     # results
#     cv_results.append(roc_auc)

# # average cross-validation result
# average_cv_result = sum(cv_results) / n_splits
# print(f'\nAverage AUC-score across {n_splits} folds: {average_cv_result}')

**#Results from XGBoost xgb_params_1:**

Fold 1, AUC Score on Validation Set: 0.7569989143865439
Fold 1, F1 Score on Validation Set: 0.862150686851945
----------------------------------------------------------------------
Fold 2, AUC Score on Validation Set: 0.7475693421069775
Fold 2, F1 Score on Validation Set: 0.8561288741919352
----------------------------------------------------------------------
Fold 3, AUC Score on Validation Set: 0.751700650628603
Fold 3, F1 Score on Validation Set: 0.8580091346466167
----------------------------------------------------------------------
Fold 4, AUC Score on Validation Set: 0.7539519677496345
Fold 4, F1 Score on Validation Set: 0.8596911149871906
----------------------------------------------------------------------
Fold 5, AUC Score on Validation Set: 0.7532216329772243
Fold 5, F1 Score on Validation Set: 0.8595033916426605
----------------------------------------------------------------------
Fold 6, AUC Score on Validation Set: 0.7574550981313459
Fold 6, F1 Score on Validation Set: 0.8636823255903835
----------------------------------------------------------------------
Fold 7, AUC Score on Validation Set: 0.7585089249091636
Fold 7, F1 Score on Validation Set: 0.862101549497247
----------------------------------------------------------------------
Fold 8, AUC Score on Validation Set: 0.7508822827948121
Fold 8, F1 Score on Validation Set: 0.8595069201886906
----------------------------------------------------------------------
Fold 9, AUC Score on Validation Set: 0.7559731707323515
Fold 9, F1 Score on Validation Set: 0.8599400101151297
----------------------------------------------------------------------
Fold 10, AUC Score on Validation Set: 0.7530089857881291
Fold 10, F1 Score on Validation Set: 0.8603447658925775
----------------------------------------------------------------------

Average AUC-score across 10 folds: 0.7539270970204786

**#Results from XGBoost xgb_params_optuna:**

Fold 1, AUC Score on Validation Set: 0.7597331514235339
Fold 1, F1 Score on Validation Set: 0.8628078396344337
----------------------------------------------------------------------
Fold 2, AUC Score on Validation Set: 0.752321894073283
Fold 2, F1 Score on Validation Set: 0.8571121897262545
----------------------------------------------------------------------
Fold 3, AUC Score on Validation Set: 0.7573406555020672
Fold 3, F1 Score on Validation Set: 0.8607201523121252
----------------------------------------------------------------------
Fold 4, AUC Score on Validation Set: 0.7547156744178872
Fold 4, F1 Score on Validation Set: 0.8588622113762977
----------------------------------------------------------------------
Fold 5, AUC Score on Validation Set: 0.7548834900735591
Fold 5, F1 Score on Validation Set: 0.8591736079816966
----------------------------------------------------------------------
Fold 6, AUC Score on Validation Set: 0.7648858974998951
Fold 6, F1 Score on Validation Set: 0.8654441741623222
----------------------------------------------------------------------
Fold 7, AUC Score on Validation Set: 0.7629786757227098
Fold 7, F1 Score on Validation Set: 0.8633988981641136
----------------------------------------------------------------------
Fold 8, AUC Score on Validation Set: 0.7535395968148548
Fold 8, F1 Score on Validation Set: 0.860076492469789
----------------------------------------------------------------------
Fold 9, AUC Score on Validation Set: 0.7577292515637706
Fold 9, F1 Score on Validation Set: 0.8594787766025463
----------------------------------------------------------------------
Fold 10, AUC Score on Validation Set: 0.7591622182763144
Fold 10, F1 Score on Validation Set: 0.8630214303703939
----------------------------------------------------------------------

Average AUC-score across 10 folds: 0.7577290505367875

In [ ]:
# Assuming trainX and trainy are your features and target variable
X_train, X_val, y_train, y_val = train_test_split(trainX, trainy, test_size=0.2, random_state=42)

xgb_pipeline.fit(X = X_train,
                y = y_train)

predictions_xgb = xgb_pipeline.predict(X_val)

cm_xgb = confusion_matrix(y_val, predictions_xgb)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_xgb, display_labels=['Not Churn', 'Churn'])
disp.plot()
plt.show()

**#Confusion Matrix from XGBoost xgb_params_1:**

* 24754 | 1298
* 3021 | 3934

# LGBM Classifier
* enable GPU Accelerator: GPU P100

In [ ]:
# #LGBM parameters
# lgbm_params_1 = {
#     'num_leaves': 31, # condition 2^max_depth > num_leaves
#     'max_depth': 5, 
#     'min_child_samples': 19, 
#     'learning_rate': 0.0617049347085071, 
#     'n_estimators': 637, 
#     'subsample': 0.7452068722516583, 
#     'colsample_bytree': 0.5621023734368561, 
#     'reg_alpha': 0.5354115365654548, 
#     'reg_lambda': 0.25902660973347336,
#     'device': 'gpu',
#     'verbosity': 0
# } 

# #LGBM Best parameters:  
# lgbm_params_optuna =  {'max_depth': 5, 
#                        'min_child_samples': 10, 
#                        'learning_rate': 0.10772545724662039, 
#                        'n_estimators': 674, 
#                        'subsample': 0.7093154192095384, 
#                        'colsample_bytree': 0.34568357689891244, 
#                        'reg_alpha': 0.7937194694663616, 
#                        'reg_lambda': 0.8145547651912818,
#                        'device': 'gpu',
#                        'verbosity': 0
#                       }
    
# # lgbm model
# lgbm_model = LGBMClassifier(**lgbm_params_optuna)

# lgbm_pipeline = make_pipeline(modelling_pipeline, lgbm_model)
# lgbm_pipeline

In [ ]:
# # folds
# n_splits = 10

# # StratifiedKFold
# stratkf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# cv_results = []


# for fold, (train_idx, val_idx) in enumerate(stratkf.split(X, y)):

#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y[train_idx], y[val_idx]

#     lgbm_pipeline.fit(X_train,y_train)

#     y_val_pred_prob = lgbm_pipeline.predict_proba(X_val)
#     y_pred = lgbm_pipeline.predict(X_val)
        
#     f1=  f1_score(y_val, y_pred, average='weighted')

#     # Evaluating the model
#     logloss = log_loss(y_val, y_val_pred_prob)
#     roc_auc = roc_auc_score(y_val, y_pred)
#     print(f'Fold {fold + 1}, AUC-Score on Validation Set: {roc_auc}')
#     print(f'Fold {fold + 1}, F1 Score on Validation Set: {f1}')
#     print('-'*70)

#     cv_results.append(roc_auc)
# average_cv_result = sum(cv_results) / n_splits
# print(f'\nAverage AUC-SCORE across {n_splits} folds: {average_cv_result}')

**#Results from LGBM lgbm_params_1:**

Fold 1, AUC-Score on Validation Set: 0.7552560082877062
Fold 1, F1 Score on Validation Set: 0.8604630349773879
----------------------------------------------------------------------
Fold 2, AUC-Score on Validation Set: 0.7503593915775693
Fold 2, F1 Score on Validation Set: 0.8566372125290799
----------------------------------------------------------------------
Fold 3, AUC-Score on Validation Set: 0.7527830068922193
Fold 3, F1 Score on Validation Set: 0.857756742748157
----------------------------------------------------------------------
Fold 4, AUC-Score on Validation Set: 0.7535119648038129
Fold 4, F1 Score on Validation Set: 0.8593270347254924
----------------------------------------------------------------------
Fold 5, AUC-Score on Validation Set: 0.7544576344467713
Fold 5, F1 Score on Validation Set: 0.8594638235981709
----------------------------------------------------------------------
Fold 6, AUC-Score on Validation Set: 0.7591516866114609
Fold 6, F1 Score on Validation Set: 0.862878698187448
----------------------------------------------------------------------
Fold 7, AUC-Score on Validation Set: 0.7607156223348945
Fold 7, F1 Score on Validation Set: 0.8621379443708285
----------------------------------------------------------------------
Fold 8, AUC-Score on Validation Set: 0.7565464696670884
Fold 8, F1 Score on Validation Set: 0.8616356267212724
----------------------------------------------------------------------
Fold 9, AUC-Score on Validation Set: 0.7560494675269485
Fold 9, F1 Score on Validation Set: 0.858651205415433
----------------------------------------------------------------------
Fold 10, AUC-Score on Validation Set: 0.7551319911436293
Fold 10, F1 Score on Validation Set: 0.8606642315432346
----------------------------------------------------------------------

Average AUC-SCORE across 10 folds: 0.7553963243292101

**#Results from LGBM lgbm_params_optuna:**

Fold 1, AUC-Score on Validation Set: 0.7583639861556996
Fold 1, F1 Score on Validation Set: 0.8615204523633316
----------------------------------------------------------------------
Fold 2, AUC-Score on Validation Set: 0.7535757811363835
Fold 2, F1 Score on Validation Set: 0.8582341247554874
----------------------------------------------------------------------
Fold 3, AUC-Score on Validation Set: 0.7566526396111933
Fold 3, F1 Score on Validation Set: 0.8602634113521627
----------------------------------------------------------------------
Fold 4, AUC-Score on Validation Set: 0.7544716781851415
Fold 4, F1 Score on Validation Set: 0.859280107948996
----------------------------------------------------------------------
Fold 5, AUC-Score on Validation Set: 0.757810368493379
Fold 5, F1 Score on Validation Set: 0.8615018421763547
----------------------------------------------------------------------
Fold 6, AUC-Score on Validation Set: 0.7644532078460704
Fold 6, F1 Score on Validation Set: 0.8661226520981758
----------------------------------------------------------------------
Fold 7, AUC-Score on Validation Set: 0.7617179132856391
Fold 7, F1 Score on Validation Set: 0.8626510429845082
----------------------------------------------------------------------
Fold 8, AUC-Score on Validation Set: 0.7570039202884369
Fold 8, F1 Score on Validation Set: 0.861769680711979
----------------------------------------------------------------------
Fold 9, AUC-Score on Validation Set: 0.758557324347017
Fold 9, F1 Score on Validation Set: 0.8608940608717451
----------------------------------------------------------------------
Fold 10, AUC-Score on Validation Set: 0.756312197899689
Fold 10, F1 Score on Validation Set: 0.8607971518717791
----------------------------------------------------------------------
Average AUC-SCORE across 10 folds: 0.757891901724865

In [ ]:
# # Assuming trainX and trainy are your features and target variable
# X_train, X_val, y_train, y_val = train_test_split(trainX, trainy, test_size=0.2, random_state=42)

# lgbm_pipeline.fit(X = X_train,
#                 y = y_train)

# predictions_lgbm = lgbm_pipeline.predict(X_val)

# cm_lgbm = confusion_matrix(y_val, predictions_lgbm)

# disp = ConfusionMatrixDisplay(confusion_matrix=cm_lgbm, display_labels=['Not Churn', 'Churn'])
# disp.plot()
# plt.show()

**#Confusion Matrix from LGBM lgbm_params_1:**

* 24751 | 1301
* 2997 | 3958

**#Confusion Matrix from LGBM lgbm_params_optuna:**

* 24733 | 1319
* 2974 | 3981

# CatBoost Classifier

In [ ]:
# #catboost parameters
# catboost_params_1 = {
#     'iterations': 848, 
#     'depth': 28,
#     'min_data_in_leaf': 5,
#     'learning_rate': 0.027876808218320774,
#     'grow_policy': 'Lossguide',
#     'bootstrap_type': 'Bernoulli',
#     'eval_metric': 'AUC',  
# }


#catboost Best parameters: {'iterations': 874, 'depth': 6, 'min_data_in_leaf': 1, 'learning_rate': 0.04892623627307684}
catboost_params_optuna = {
    'iterations': 823, 
    'depth': 15, 
    'min_data_in_leaf': 5, 
    'learning_rate': 0.03517275419501559,
    'grow_policy': 'Lossguide',
    'bootstrap_type': 'Bernoulli',
    'eval_metric': 'AUC',  
}



# catboost model
cb_model = CatBoostClassifier(**catboost_params_optuna, random_state=42, verbose=0)


cb_pipeline = make_pipeline(modelling_pipeline, cb_model)
cb_pipeline

In [ ]:
# n_splits = 10

# stratkf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)


# cv_results = []


# for fold, (train_idx, val_idx) in enumerate(stratkf.split(X, y)):
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y[train_idx], y[val_idx]

    
#     cb_pipeline.fit(X_train,y_train)

#     y_val_pred_prob = cb_pipeline.predict_proba(X_val)
#     y_pred = cb_pipeline.predict(X_val)
        
#     f1=  f1_score(y_val, y_pred, average='weighted')

#     # Evaluating the model
#     logloss = log_loss(y_val, y_val_pred_prob)
#     roc_auc = roc_auc_score(y_val, y_pred)
#     print(f'Fold {fold + 1}, AUC- score on Validation Set: {roc_auc}')
#     print(f'Fold {fold + 1}, F1 Score on Validation Set: {f1}')
#     print(f'Fold {fold + 1}, Log Loss Score on Validation Set: {logloss}')
#     print('-'*70)

 
#     cv_results.append(logloss)

# average_cv_result = sum(cv_results) / n_splits
# print(f'\nAverage Logarithmic Loss across {n_splits} folds: {average_cv_result}')

**#Results from CatBoost catboost_params_1:**

Fold 1, AUC- score on Validation Set: 0.7601764377159651
Fold 1, F1 Score on Validation Set: 0.8622805283515061
----------------------------------------------------------------------
Fold 2, AUC- score on Validation Set: 0.7509179560747344
Fold 2, F1 Score on Validation Set: 0.8562851720930656
----------------------------------------------------------------------
Fold 3, AUC- score on Validation Set: 0.7573649083813373
Fold 3, F1 Score on Validation Set: 0.8601180215319723
----------------------------------------------------------------------
Fold 4, AUC- score on Validation Set: 0.754576392593261
Fold 4, F1 Score on Validation Set: 0.859300233416504
----------------------------------------------------------------------
Fold 5, AUC- score on Validation Set: 0.7547961003655115
Fold 5, F1 Score on Validation Set: 0.8589244861930165
----------------------------------------------------------------------
Fold 6, AUC- score on Validation Set: 0.7613447159831187
Fold 6, F1 Score on Validation Set: 0.86366368747521
----------------------------------------------------------------------
Fold 7, AUC- score on Validation Set: 0.764113619430136
Fold 7, F1 Score on Validation Set: 0.8638403970008023
----------------------------------------------------------------------
Fold 8, AUC- score on Validation Set: 0.7569059989155357
Fold 8, F1 Score on Validation Set: 0.8613771465696838
----------------------------------------------------------------------
Fold 9, AUC- score on Validation Set: 0.7603728975297402
Fold 9, F1 Score on Validation Set: 0.8607626425505464
----------------------------------------------------------------------
Fold 10, AUC- score on Validation Set: 0.7567380535264768
Fold 10, F1 Score on Validation Set: 0.8605048826043739
----------------------------------------------------------------------

Average Logarithmic Loss across 10 folds: 0.31481146006637484

**#Results from CatBoost catboost_params_optuna:**

Fold 1, AUC- score on Validation Set: 0.7586050624166114
Fold 1, F1 Score on Validation Set: 0.8619869952084535
Fold 1, Log Loss Score on Validation Set: 0.31187725431210056
----------------------------------------------------------------------
Fold 2, AUC- score on Validation Set: 0.7505304822159051
Fold 2, F1 Score on Validation Set: 0.8566246561650627
Fold 2, Log Loss Score on Validation Set: 0.3167632088514108
----------------------------------------------------------------------
Fold 3, AUC- score on Validation Set: 0.7547002168057753
Fold 3, F1 Score on Validation Set: 0.8585425371928028
Fold 3, Log Loss Score on Validation Set: 0.3125097063652024
----------------------------------------------------------------------
Fold 4, AUC- score on Validation Set: 0.755529394923008
Fold 4, F1 Score on Validation Set: 0.8596222334125978
Fold 4, Log Loss Score on Validation Set: 0.3155877979623915
----------------------------------------------------------------------
Fold 5, AUC- score on Validation Set: 0.7562248081916412
Fold 5, F1 Score on Validation Set: 0.8605472605780414
Fold 5, Log Loss Score on Validation Set: 0.3175418904047049
----------------------------------------------------------------------
Fold 6, AUC- score on Validation Set: 0.7616068851072618
Fold 6, F1 Score on Validation Set: 0.8644184663470575
Fold 6, Log Loss Score on Validation Set: 0.30547185380441755
----------------------------------------------------------------------
Fold 7, AUC- score on Validation Set: 0.7644526465974734
Fold 7, F1 Score on Validation Set: 0.864704019389124
Fold 7, Log Loss Score on Validation Set: 0.31440807258735987
----------------------------------------------------------------------
Fold 8, AUC- score on Validation Set: 0.7538643946795217
Fold 8, F1 Score on Validation Set: 0.8602792340602818
Fold 8, Log Loss Score on Validation Set: 0.30930159191729895
----------------------------------------------------------------------
Fold 9, AUC- score on Validation Set: 0.7583999106228116
Fold 9, F1 Score on Validation Set: 0.8601638286761754
Fold 9, Log Loss Score on Validation Set: 0.315381648206708
----------------------------------------------------------------------
Fold 10, AUC- score on Validation Set: 0.7561690134781539
Fold 10, F1 Score on Validation Set: 0.8607231859572884
Fold 10, Log Loss Score on Validation Set: 0.3158966435517018
----------------------------------------------------------------------

Average Logarithmic Loss across 10 folds: 0.31347396679632966

In [ ]:
# Assuming trainX and trainy are your features and target variable
X_train, X_val, y_train, y_val = train_test_split(trainX, trainy, test_size=0.2, random_state=42)

cb_pipeline.fit(X = X_train,
                y = y_train)

predictions_cb = cb_pipeline.predict(X_val)

cm_cb = confusion_matrix(y_val, predictions_cb)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_cb, display_labels=['Not Churn', 'Churn'])
disp.plot()
plt.show()

**#Confusion Matrix from catboost catboost_params_1:**

* 24699 | 1353
* 2967 | 3988

# <p style="background-color:red ;color:white;font-family:cursive ;font-size:110%;text-align:center;border-radius: 15px 50px;">6. Hyperparameter Tuning with Optuna</p>

## Hyperparameter tuning for XGBoost

In [ ]:
# from sklearn.model_selection import  cross_val_score

# def objective_xgb(trial):
#     """Define the objective function for XGBClassifier"""

#     params = {
#         'max_depth': trial.suggest_int('max_depth', 5, 10),
#         'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0),
#         'n_estimators': trial.suggest_int('n_estimators', 150, 1000),
#         'subsample': trial.suggest_float('subsample', 0.01, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.01, 1.0),
#         'random_state': trial.suggest_categorical('random_state', [42]),
#         'tree_method': 'hist',  # Use GPU for training
#         'device' : 'cuda',
#         'eval_metric': 'auc',  # Evaluation metric
#         'verbosity': 2,  # Set verbosity to 0 for less output
#     }

#     xgb_model = xgb.XGBClassifier(**params)
#     xgb_pipeline = make_pipeline(modelling_pipeline, xgb_model)

#     # Assuming 'trainX' and 'trainy' are your training data
#     cv = abs(cross_val_score(xgb_pipeline, trainX, trainy, cv=skf, scoring='roc_auc').mean())

    
#     return cv


# # Assuming 'skf' is your StratifiedKFold object
# skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# study = optuna.create_study(direction='maximize')
# study.optimize(objective_xgb, n_trials=50)

# # Get the best parameters
# best_params_xgb = study.best_params
# print("Best Hyperparameters for XGBoost:", best_params_xgb)

* **Best Hyperparameters for XGBoost: {'max_depth': 5, 'min_child_weight': 2, 'learning_rate': 0.07353564842520434, 'n_estimators': 463, 'subsample': 0.8131149969184862, 'colsample_bytree': 0.6598001508811656, 'random_state': 42}**

## Hyperparameter tuning for LGBM

In [ ]:
# # Assuming 'skf' is your StratifiedKFold object
# skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# def objective_lgbm(trial):
#     """Define the objective function for LGBMClassifier"""

#     params = {
#         'objective': 'binary',  # Objective for binary classification
#         'boosting_type': 'gbdt',
#         'device': 'gpu',
#         'metric': 'auc',  # Evaluation metric
#         'max_depth': trial.suggest_int('max_depth', 5, 10),
#         'min_child_samples': trial.suggest_int('min_child_samples', 1, 20),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0),
#         'n_estimators': trial.suggest_int('n_estimators', 150, 1000),
#         'subsample': trial.suggest_float('subsample', 0.1, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.1, 1.0),
#         'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
#         'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
#         'random_state': 42,
#     }

#     lgb_model = lgb.LGBMClassifier(**params)
#     lgb_pipeline = make_pipeline(modelling_pipeline, lgb_model)

#     # Assuming 'trainX' and 'trainy' are your training data
#     cv = abs(cross_val_score(lgb_pipeline, trainX, trainy, cv=skf, scoring='roc_auc').mean())

#     return cv

# # Create an Optuna study
# study = optuna.create_study(direction='maximize')
# study.optimize(objective_lgbm, n_trials=50)

# # Get the best parameters
# best_params_lgb = study.best_params
# print("Best Hyperparameters for LGBM:", best_params_lgb)


* **Best Hyperparameters for LGBM:**  {'max_depth': 5, 'min_child_samples': 19, 'learning_rate': 0.0617049347085071, 'n_estimators': 637, 'subsample': 0.7452068722516583, 'colsample_bytree': 0.5621023734368561, 'reg_alpha': 0.5354115365654548, 'reg_lambda': 0.25902660973347336}

## Hyperparameter tuning for CatBoost

In [ ]:
# #catboost parameters
# catboost_params_1 = {
#     'iterations': 848, 
#     'depth': 28,
#     'min_data_in_leaf': 5,
#     'learning_rate': 0.027876808218320774,
#     'grow_policy': 'Lossguide',
#     'bootstrap_type': 'Bernoulli',
#     'eval_metric': 'AUC',  
# }

In [ ]:
# # Suppress FutureWarnings related to is_sparse
# warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.utils.validation")

# # Assuming 'skf' is your StratifiedKFold object
# skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# # Assuming 'skf' is your StratifiedKFold object
# skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# def objective(trial):
#     params = {
#         'iterations': trial.suggest_int('iterations', 500, 1000),
#         'depth': trial.suggest_int('depth', 5, 15),
#         'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 10),
#         'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.2, log=True),
#     }
    
#     cb_model = CatBoostClassifier(**params, random_state=42, grow_policy='Lossguide', verbose=0)
#     cb_pipeline = make_pipeline(modelling_pipeline, cb_model)

#     # Using cross_val_score with 'cv' parameter
#     cv = cross_val_score(cb_pipeline, trainX, trainy, cv=skf, scoring='roc_auc', n_jobs=-1).mean()

#     return cv

# # Create an Optuna study with pruning
# study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())

# # Perform the optimization with more trials
# study.optimize(objective, n_trials=50)

# # Get the best hyperparameters
# best_params_cb = study.best_params
# print("Best Hyperparameters for CatBoost:", best_params_cb)

* Best Hyperparameters for CatBoost: {'iterations': 874, 'depth': 6, 'min_data_in_leaf': 1, 'learning_rate': 0.04892623627307684}
* Trial 1 finished with value: 0.8940361824097615 and parameters: {'iterations': 874, 'depth': 6, 'min_data_in_leaf': 1, 'learning_rate': 0.04892623627307684}. Best is trial 1 with value: 0.8940361824097615.



# <p style="background-color:red ;color:white;font-family:cursive ;font-size:110%;text-align:center;border-radius: 15px 50px;">7. Setting up the Ensemble</p>


In [ ]:
from sklearn.ensemble import VotingClassifier

In [ ]:
# Ensemble using a VotingClassifier
# {'weight_xgb': 0.3745401188473625, 'weight_lgb': 0.9507143064099162, 'weight_cb': 0.7319939418114051}. Best is trial 0 with value: 0.894922718883933.
# {'weight_xgb': 0.23101998962575393, 'weight_lgb': 0.5189511254853938, 'weight_cb': 0.3670717954717113}. Best is trial 11 with value: 0.8951460116162566.

ensemble_model = VotingClassifier(estimators=[
    ('xgb', xgb_pipeline),
#    ('lgb', lgbm_pipeline),
    ('cb', cb_pipeline)
        
]
                                  , voting='soft',
                                  weights = [1,1]) 

ensemble_model

In [ ]:
# # number of folds
# n_splits = 10

# #  StratifiedKFold
# stratkf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# #  cross-validation results
# cv_results = []

# # stratified k-fold cross-validation
# for fold, (train_idx, val_idx) in enumerate(stratkf.split(X, y)):
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y[train_idx], y[val_idx]

#     ensemble_model.fit(X_train, y_train )

#     # predictions on the validation set
#     y_val_pred_prob = ensemble_model.predict(X_val)
#     y_pred = ensemble_model.predict(X_val)
        
#     f1=  f1_score(y_val, y_pred, average='weighted')

#     # Evaluating the model
    
#     roc_auc = roc_auc_score(y_val, y_val_pred_prob)
#     print(f'Fold {fold + 1}, AUC Score on Validation Set: {roc_auc}')
#     print(f'Fold {fold + 1}, F1 Score on Validation Set: {f1}')
#     print('-'*70)

#     # results
#     cv_results.append(roc_auc)

# # average cross-validation result
# average_cv_result = sum(cv_results) / n_splits
# print(f'\nAverage AUC-score across {n_splits} folds: {average_cv_result}')

*** Ensemble Model (baseline 1,1,1) Results:**

Fold 1, AUC Score on Validation Set: 0.7614829240362847
Fold 1, F1 Score on Validation Set: 0.8641206229401505
----------------------------------------------------------------------
Fold 2, AUC Score on Validation Set: 0.7515675458973635
Fold 2, F1 Score on Validation Set: 0.856688605977892
----------------------------------------------------------------------
Fold 3, AUC Score on Validation Set: 0.7541521721600538
Fold 3, F1 Score on Validation Set: 0.8590412579437448
----------------------------------------------------------------------
Fold 4, AUC Score on Validation Set: 0.7557493963959189
Fold 4, F1 Score on Validation Set: 0.8598038536226197
----------------------------------------------------------------------
Fold 5, AUC Score on Validation Set: 0.7557078652189886
Fold 5, F1 Score on Validation Set: 0.8600757772751055
----------------------------------------------------------------------
Fold 6, AUC Score on Validation Set: 0.7639257332085644
Fold 6, F1 Score on Validation Set: 0.8655059960363871
----------------------------------------------------------------------
Fold 7, AUC Score on Validation Set: 0.7632476788738897
Fold 7, F1 Score on Validation Set: 0.8637776146854188
----------------------------------------------------------------------
Fold 8, AUC Score on Validation Set: 0.7573776788395545
Fold 8, F1 Score on Validation Set: 0.862168271476127
----------------------------------------------------------------------
Fold 9, AUC Score on Validation Set: 0.7598912471894651
Fold 9, F1 Score on Validation Set: 0.8612356762440299
----------------------------------------------------------------------
Fold 10, AUC Score on Validation Set: 0.7570281200073635
Fold 10, F1 Score on Validation Set: 0.861166780527565
----------------------------------------------------------------------

Average AUC-score across 10 folds: 0.7580130361827446

*** Ensemble Model {'weight_xgb': 0.3745401188473625, 'weight_lgb': 0.9507143064099162, 'weight_cb': 0.7319939418114051} Results:**


Fold 1, AUC Score on Validation Set: 0.7618530115297572
Fold 1, F1 Score on Validation Set: 0.8640007358998513
----------------------------------------------------------------------
Fold 2, AUC Score on Validation Set: 0.7525419306313073
Fold 2, F1 Score on Validation Set: 0.8572935205852342
----------------------------------------------------------------------
Fold 3, AUC Score on Validation Set: 0.7555524568210716
Fold 3, F1 Score on Validation Set: 0.8593557820513934
----------------------------------------------------------------------
Fold 4, AUC Score on Validation Set: 0.7558646834607102
Fold 4, F1 Score on Validation Set: 0.8599655467086817
----------------------------------------------------------------------
Fold 5, AUC Score on Validation Set: 0.7573597518990672
Fold 5, F1 Score on Validation Set: 0.8609960557006606
----------------------------------------------------------------------
Fold 6, AUC Score on Validation Set: 0.7630666266793549
Fold 6, F1 Score on Validation Set: 0.8650651750166936
----------------------------------------------------------------------
Fold 7, AUC Score on Validation Set: 0.7637819875384324
Fold 7, F1 Score on Validation Set: 0.8640161430232965
----------------------------------------------------------------------
Fold 8, AUC Score on Validation Set: 0.7574650685476022
Fold 8, F1 Score on Validation Set: 0.8624189361923639
----------------------------------------------------------------------
Fold 9, AUC Score on Validation Set: 0.7609877618752939
Fold 9, F1 Score on Validation Set: 0.8616251841374007
----------------------------------------------------------------------
Fold 10, AUC Score on Validation Set: 0.7589558438656584
Fold 10, F1 Score on Validation Set: 0.8620918477700645
----------------------------------------------------------------------

Average AUC-score across 10 folds: 0.7587429122848255

In [ ]:
# # Assuming trainX and trainy are your features and target variable
# X_train, X_val, y_train, y_val = train_test_split(trainX, trainy, test_size=0.2, random_state=42)

# ensemble_model.fit(X = X_train, y = y_train)

# predictions_ensemble = ensemble_model.predict(X_val)

# cm_ensemble = confusion_matrix(y_val, predictions_ensemble)

# disp = ConfusionMatrixDisplay(confusion_matrix=cm_ensemble, display_labels=['Not Churn', 'Churn'])
# disp.plot()
# plt.show()

**#Confusion Matrix from Ensemble Model (baseline 1,1,1):**

* 24739 | 1313
* 2970 | 3985

**#Confusion Matrix from Ensemble Model {'weight_xgb': 0.3745401188473625, 'weight_lgb': 0.9507143064099162, 'weight_cb': 0.7319939418114051}:**

* 24717 | 1335
* 2957 | 3998

# <p style="background-color:red ;color:white;font-family:cursive ;font-size:110%;text-align:center;border-radius: 15px 50px;">8. Hyperparameter Tuning for Ensemble Weight</p>


In [ ]:
# # Define the parameter search space
# def objective(trial):
#     weights = [
#         trial.suggest_float('weight_xgb', 0, 1),  # Adjust the range based on your expectations
#         trial.suggest_float('weight_lgb', 0, 1),
#         trial.suggest_float('weight_cb', 0, 1)
#     ]

#     ensemble_model = VotingClassifier(
#         estimators=[
#             ('xgb', xgb_pipeline),
#             ('lgb', lgbm_pipeline),
#             ('cb', cb_pipeline)
#         ], voting='soft', weights=weights)

#     # Assuming 'trainX' and 'trainy' are your training data
#     cv = abs(cross_val_score(ensemble_model, trainX, trainy, cv=skf, scoring='roc_auc').mean())

#     return cv

# # Assuming 'skf' is your StratifiedKFold object
# skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# # Use the 'sampler' parameter for parallelization
# study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
# study.optimize(objective, n_trials=50)

# best_weights = study.best_params
# print("Best Weights for Ensemble:", best_weights)


* Trial 1 finished with value: 0.8947827272920067 and parameters: {'weight_xgb': 0.5986584841970366, 'weight_lgb': 0.15601864044243652, 'weight_cb': 0.15599452033620265}. Best is trial 0 with value: 0.8949226492021898.

_____

* Trial 0 finished with value: 0.894922718883933 and parameters: {'weight_xgb': 0.3745401188473625, 'weight_lgb': 0.9507143064099162, 'weight_cb': 0.7319939418114051}. Best is trial 0 with value: 0.894922718883933.
* Trial 1 finished with value: 0.8948201194071821 and parameters: {'weight_xgb': 0.5986584841970366, 'weight_lgb': 0.15601864044243652, 'weight_cb': 0.15599452033620265}. Best is trial 0 with value: 0.894922718883933.
* Trial 2 finished with value: 0.8948752164576543 and parameters: {'weight_xgb': 0.05808361216819946, 'weight_lgb': 0.8661761457749352, 'weight_cb': 0.6011150117432088}. Best is trial 0 with value: 0.894922718883933.
* Trial 3 finished with value: 0.8944471339057675 and parameters: {'weight_xgb': 0.7080725777960455, 'weight_lgb': 0.020584494295802447, 'weight_cb': 0.9699098521619943}. Best is trial 0 with value: 0.894922718883933.
* Trial 4 finished with value: 0.894827628230162 and parameters: {'weight_xgb': 0.8324426408004217, 'weight_lgb': 0.21233911067827616, 'weight_cb': 0.18182496720710062}. Best is trial 0 with value: 0.894922718883933.
* Trial 5 finished with value: 0.8947256695087843 and parameters: {'weight_xgb': 0.18340450985343382, 'weight_lgb': 0.3042422429595377, 'weight_cb': 0.5247564316322378}. Best is trial 0 with value: 0.894922718883933.
* Trial 6 finished with value: 0.8948473714675028 and parameters: {'weight_xgb': 0.43194501864211576, 'weight_lgb': 0.2912291401980419, 'weight_cb': 0.6118528947223795}. Best is trial 0 with value: 0.894922718883933.

_____

* Trial 0 finished with value: 0.8950952656262812 and parameters: {'weight_xgb': 0.3745401188473625, 'weight_lgb': 0.9507143064099162, 'weight_cb': 0.7319939418114051}. Best is trial 0 with value: 0.8950952656262812.
* Trial 7 finished with value: 0.8951443050200905 and parameters: {'weight_xgb': 0.13949386065204183, 'weight_lgb': 0.29214464853521815, 'weight_cb': 0.3663618432936917}. Best is trial 7 with value: 0.8951443050200905.
* Trial 8 finished with value: 0.8950380903749693 and parameters: {'weight_xgb': 0.45606998421703593, 'weight_lgb': 0.7851759613930136, 'weight_cb': 0.19967378215835974}. Best is trial 7 with value: 0.8951443050200905.
* Trial 9 finished with value: 0.8950655663325969 and parameters: {'weight_xgb': 0.5142344384136116, 'weight_lgb': 0.5924145688620425, 'weight_cb': 0.046450412719997725}. Best is trial 7 with value: 0.8951443050200905.
* Trial 10 finished with value: 0.8951450434980701 and parameters: {'weight_xgb': 0.20761884162151767, 'weight_lgb': 0.4733273214546686, 'weight_cb': 0.3534938756581689}. Best is trial 10 with value: 0.8951450434980701.
* Trial 11 finished with value: 0.8951460116162566 and parameters: {'weight_xgb': 0.23101998962575393, 'weight_lgb': 0.5189511254853938, 'weight_cb': 0.3670717954717113}. Best is trial 11 with value: 0.8951460116162566.
* Trial 12 finished with value: 0.8949843151526515 and parameters: {'weight_xgb': 0.2554434587651989, 'weight_lgb': 0.5042867903597874, 'weight_cb': 0.3577123916785768}. Best is trial 11 with value: 0.8951460116162566.
* Trial 13 finished with value: 0.8949577333736858 and parameters: {'weight_xgb': 0.00876296378757771, 'weight_lgb': 0.6303855696153478, 'weight_cb': 0.3560912233349937}. Best is trial 11 with value: 0.8951460116162566.
* Trial 14 finished with value: 0.8950268266746502 and parameters: {'weight_xgb': 0.2986659104965187, 'weight_lgb': 0.4343562717564112, 'weight_cb': 0.4328664463880069}. Best is trial 11 with value: 0.8951460116162566.

In [ ]:
# # Define the parameter grid for weights
# param_grid = {'weights': [(round(w1, 2), round(w2, 2), round(w3, 2))
#                            for w1 in np.arange(0.0, 1.0, 0.1)
#                            for w2 in np.arange(0.0, 1.0, 0.1)
#                            for w3 in np.arange(0.0, 1.0, 0.1)
#                            if round(w1 + w2 + w3, 2) == 1.0]}

# # Extract weights from the parameter grid
# weights_combinations = param_grid['weights']

# # Display the combinations
# for combination in weights_combinations:
#     print(combination)

In [ ]:
# import itertools
# import numpy as np

# # Define the possible values for weights
# weights_range = np.arange(0.0, 1.1, 0.1)

# # Generate all possible combinations of weights
# weight_combinations = itertools.product(weights_range, repeat=3)

# # Filter combinations where the sum is 1.0
# valid_weight_combinations = [weights for weights in weight_combinations if round(sum(weights), 2) == 1.0]

# # Display the combinations
# for combination in valid_weight_combinations:
#     print(combination)

In [ ]:
# # Define the parameter grid for weights
# param_grid = {'weights': [(round(w1, 2), round(w2, 2), round(w3, 2))
#                            for w1 in np.arange(0.0, 1.0, 0.1)
#                            for w2 in np.arange(0.0, 1.0, 0.1)
#                            for w3 in np.arange(0.0, 1.0, 0.1)
#                            if round(w1 + w2 + w3, 2) == 1.0]}

# # Create the VotingClassifier
# ensemble_model = VotingClassifier(estimators=[
#     ('xgb', xgb_pipeline),
#     ('lgb', lgbm_pipeline),
#     ('cb', cb_pipeline)
# ], voting='soft')

# # Create a StratifiedKFold instance
# stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

# # Perform grid search
# grid_search = GridSearchCV(estimator=ensemble_model,
#                            param_grid=param_grid,
#                            scoring='roc_auc',
#                            cv=stratified_kfold,
#                            verbose=2)
                           
# # Fit the grid search
# grid_search.fit(trainX, trainy)

# # Get the best weights
# best_weights = grid_search.best_params_['weights']
# best_weights

# <p style="background-color:red ;color:white;font-family:cursive ;font-size:110%;text-align:center;border-radius: 15px 50px;">9. Review of Features</p>

## XGB Feature Importance

In [ ]:
# Access individual base models' feature importances (if available)
xgb_feature_importance = xgb_pipeline.named_steps['xgbclassifier'].feature_importances_

In [ ]:
train_X = modelling_pipeline.fit_transform(train.drop(['Exited'], axis=1))


sorted_idx = np.argsort(xgb_feature_importance)
fig = plt.figure(figsize=(18, 16))
plt.barh(range(len(sorted_idx)), xgb_feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), np.array(train_X.columns)[sorted_idx])
plt.title('XGB_Feature Importance')
plt.show()


## LGBM Feature Importance

In [ ]:
# # Access individual base models' feature importances (if available)
# lgbm_feature_importance = lgbm_pipeline.named_steps['lgbmclassifier'].feature_importances_

In [ ]:
# sorted_idx = np.argsort(lgbm_feature_importance)
# fig = plt.figure(figsize=(18, 16))
# plt.barh(range(len(sorted_idx)), lgbm_feature_importance[sorted_idx], align='center')
# plt.yticks(range(len(sorted_idx)), np.array(train_X.columns)[sorted_idx])
# plt.title('LGBM_Feature Importance')
# plt.show()


## CatBoost Feature Importance

In [ ]:
# Access individual base models' feature importances (if available)
cb_feature_importance = cb_pipeline.named_steps['catboostclassifier'].feature_importances_

In [ ]:
sorted_idx = np.argsort(cb_feature_importance)
fig = plt.figure(figsize=(18, 16))
plt.barh(range(len(sorted_idx)), cb_feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), np.array(train_X.columns)[sorted_idx])
plt.title('CB_Feature Importance')
plt.show()

# <p style="background-color:red ;color:white;font-family:cursive ;font-size:110%;text-align:center;border-radius: 15px 50px;">10. Submission</p>

In [ ]:
# # Fit the ensemble baseline model
# ensemble_model.fit(X=train.drop(['Exited'], axis=1), y=train['Exited'])

# # Create submission file with probability predictions
# predictions = ensemble_model.predict_proba(test)[:, 1]  # Use the probabilities of class 1

# sample['Exited'] = predictions
# sample.to_csv('submission_ensemble_baseline.csv', index=False)


In [ ]:
# Fit the ensemble final model
ensemble_model.fit(X=train.drop(['Exited'], axis=1), y=train['Exited'])

# Create submission file with probability predictions
predictions = ensemble_model.predict_proba(test)[:, 1]  # Use the probabilities of class 1

sample['Exited'] = predictions
sample.to_csv('submission_ensemble_final.csv', index=False)

In [ ]:
sample.head()